# WikiLens · question to cross-edition evidence

**Restart the kernel first.** Select **Junction T4 Python 3.12** on an authorized running job. The previous T4 job ended with exit 137 (OOMKilled); this notebook does not create or pay for a replacement. Recreate the project environment with `uv sync --locked` when needed.

Run one cell at a time. Each step exposes its inputs, raw outputs and evidence for review. The runtime loads **one GPU model at a time**, switches between Qwen and BGE, and keeps only CPU results in notebook variables. No training occurs. Existing old model variables require a kernel restart.

Flow: question → retrieval-only HyDE → direct + HyDE vectors → MariaDB candidates per language → relational edition alignment → full-article passages → exact quotations → cited answer draft. The balanced retrieval/fusion policy is experimental: the saved development measurement improves edition coverage but lowers first-topic accuracy. Inspect candidate relevance before continuing. Dataset M, time travel and deep search remain deferred.

Citation and quotation checks establish provenance, not whether a claim follows from the evidence. All answer claims require manual review before use in the pitch. This notebook is newly prepared and unexecuted; GPU memory and answer quality are not yet measured for this version.


## 1. Setup and question

Use the existing project database; no import or database writes are needed. Start with one question. Changing it requires rerunning the subsequent cells so evidence and answer stay aligned.


In [ ]:
from pathlib import Path
import sys
import json
import html
import subprocess
from datetime import datetime, timezone

project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
assert sys.version_info[:2] == (3, 12)
assert Path(sys.prefix).resolve() == (project_dir / ".venv").resolve(), "Select the project's .venv kernel."

import pandas as pd
from IPython.display import HTML, display
from wikilens.database import connect
from wikilens.torch_runtime import TorchRuntime
from wikilens.pipeline import generate_hyde, retrieve, rank_evidence, select_quotes, synthesize, begin_stage

question = "When did Finland declare its independence?"
runtime = TorchRuntime(context_tokens=4096, max_batch_tokens=1024)
run_record = {"question": question, "dataset_release": "s-2026-10",
              "created_at": datetime.now(timezone.utc).isoformat(), "stages": {},
              "claim_support_reviewed": False}
report_dir = project_dir / "reports"
report_dir.mkdir(exist_ok=True)
with connect() as db, db.cursor() as cursor:
    cursor.execute("SELECT lang,COUNT(*) AS articles FROM article GROUP BY lang")
    display(pd.DataFrame(cursor.fetchall()))
print("Question:", question)
print("Models load only when a stage needs them. No job is created here.")


## 2. Generate HyDE and inspect it

Qwen produces a short hypothetical English passage with thinking disabled. It may invent facts. This text is only a retrieval input; it never enters the final evidence context. The raw output, prompt and configuration are retained. The runtime uses bounded greedy generation for repeatability.


In [ ]:
begin_stage(run_record, "hyde", question)
hyde = run_record["stages"]["hyde"] = generate_hyde(runtime, question)
print("Status:", hyde["status"])
print("HYPOTHETICAL — NOT EVIDENCE:\n", hyde["raw_output"])
if hyde["status"] != "completed":
    print(hyde.get("error"))


## 3. MariaDB retrieval across three languages

Embed the original question and HyDE separately with BGE-M3. Search each language with native cosine SQL, fuse branch ranks within each language, then fetch selected topic counterparts through Wikidata. Distances across different embedding inputs are not compared. `origin` distinguishes a vector candidate from a relationally joined counterpart. EXPLAIN records the actual filtered query shape.

This policy favors edition discovery, not agreement or guaranteed relevance. Review the candidates against the original question. The local 37-question cached-vector measurement is a development diagnostic, not a benchmark of this new direct+HyDE branch fusion.


In [ ]:
begin_stage(run_record, "retrieval", question)
hyde = run_record["stages"]["hyde"]
assert hyde["status"] == "completed", "Inspect HyDE failure before continuing."
retrieval = run_record["stages"]["retrieval"] = retrieve(question, hyde, runtime)
display(pd.DataFrame(retrieval["candidates"])[["lang", "title", "wikidata_id", "language_rank"]])
display(pd.DataFrame(retrieval["articles"])[["lang", "title", "origin", "truncated"]])
print("Aligned topics:", retrieval["aligned_topics"])
print("SQL capture time:", retrieval["retrieval_seconds"], "seconds (includes EXPLAIN and hydration).")


## 4. Rank literal passages beyond the introductions

Process full article texts into 700-character overlapping slices with stable IDs and exact source offsets. BGE scores against the original question in batches of one, with a 1,024-token limit and no silent truncation. At most 3,000 candidates are accepted; inspect scope if the bound fails. Select up to six passages, including one per available language and at most two per article. Language seeding is a discovery policy, not relevance verification. The dedicated cross-encoder and Qwen relevance-grading experiments remain deferred/archived.


In [ ]:
begin_stage(run_record, "ranking", question)
retrieval = run_record["stages"]["retrieval"]
ranking = run_record["stages"]["ranking"] = rank_evidence(runtime, question, retrieval["articles"])
display(pd.DataFrame([
    {"id": p["id"], "lang": p["lang"], "title": p["title"],
     "score": p["query_similarity"], "start": p["start_char"], "passage": p["text"]}
    for p in ranking["selected"]
]))
print("Ranked", len(ranking["ranked"]), "passages; selected", len(ranking["selected"]))


## 5. Select quotations and verify exact provenance

Qwen selects up to three quotations using source IDs. Python validates the complete response, source IDs, quote lengths and literal occurrences, and preserves exact article offsets. Invalid model output is rejected in full, and raw output stays available for diagnosis. Read the quotations before continuing: valid provenance still does not prove relevance.


In [ ]:
begin_stage(run_record, "quotations", question)
ranking = run_record["stages"]["ranking"]
quotations = run_record["stages"]["quotations"] = select_quotes(runtime, question, ranking["selected"])
print("Status:", quotations["status"])
for source in quotations["sources"]:
    print(source["id"], source["lang"], source["title"])
    print(source["quote"])
    print(source["revision_url"], "\n")
if quotations["status"] in {"failed", "rejected"}:
    print(quotations.get("error", quotations.get("validation_error")))
    print("Raw output:", quotations["raw_output"])


## 6. Fluent answer synthesis with citations

Generate a short English answer from verified quotations only. Python checks source membership per sentence, and the renderer builds links from stored revision metadata. A rejection or insufficient evidence produces no synthesized answer. **AI draft: manually check every claim.** The matrix records cited versus not cited; neither label is an entailment judgment.


In [ ]:
begin_stage(run_record, "synthesis", question)
quotations = run_record["stages"]["quotations"]
assert quotations["status"] in {"completed", "insufficient_evidence"}, "Inspect rejected quotations first."
synthesis = run_record["stages"]["synthesis"] = synthesize(runtime, question, quotations["sources"])
run_record["status"] = synthesis["status"]
print("Status:", synthesis["status"])
source_lookup = {s["id"]: s for s in quotations["sources"]}
if synthesis["status"] == "draft":
    parts = []
    for sentence in synthesis["sentences"]:
        links = " ".join(
            '<a href="' + html.escape(source_lookup[s]["revision_url"], quote=True)
            + '" target="_blank" rel="noopener noreferrer">[' + html.escape(s) + ']</a>'
            for s in sentence["sources"]
        )
        parts.append(html.escape(sentence["text"]) + " " + links)
    display(HTML("<strong>AI draft — review claim support.</strong><p>" + " ".join(parts) + "</p>"))
    display(pd.DataFrame(synthesis["claim_matrix"]))
else:
    print(synthesis.get("validation_error", synthesis.get("error", "Insufficient evidence.")))


## 7. Save the run and release GPU memory

Run even after a failed stage to preserve all completed stages and diagnostics. The runtime record includes model revisions, bounds, elapsed times and CUDA allocation peaks. This cell releases the runtime's model. Copy reports locally before the job expires. Use a separate human claim-review file before exporting a reviewed recording fixture; do not relabel a draft as verified automatically.


In [ ]:
assert question == run_record["question"], "Question changed; rerun Setup first."
runtime.close()
run_record["runtime"] = runtime.stats()
run_record["status"] = run_record["stages"].get("synthesis", {}).get("status", "incomplete")
run_record["seconds"] = sum(
    stage.get("seconds", stage.get("embedding_and_retrieval_seconds", 0.0))
    for stage in run_record["stages"].values()
)
run_record["timing_basis"] = "Sum of measured stage times; excludes interactive pauses, setup and cleanup."
run_record["code_commit"] = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=project_dir, text=True
).strip()
run_record["working_tree_changes"] = bool(subprocess.check_output(
    ["git", "status", "--porcelain"], cwd=project_dir, text=True
).strip())
path = report_dir / ("pipeline-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
path.write_text(json.dumps(run_record, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", path)
print("GPU model released. Download this file before the temporary job expires.")
